In [1]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
import gymnasium as gym
from env.user_sim import UserSimEnv
from morl_baselines.multi_policy.morld import morld
from morl_baselines.common.pareto import ParetoArchive
from morl_baselines.common.weights import equally_spaced_weights, random_weights
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from ipywidgets import interact
import value_iteration
import utils
from morld.vi_morld_simple import VIMORLD
from morld.mo_pi import MOPolicyIteration

In [2]:
MAX_COUNT = 3
NUM_VALS = 3
NUM_STOCHASTIC_STATES = NUM_VALS**3
NUM_STATES = NUM_STOCHASTIC_STATES * (MAX_COUNT+1)**4 
NUM_ACTIONS = 104
NUM_OBJECTIVES = 6

In [3]:
data_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\'
results_folder = 'C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\'

action_file = 'normalized_challenges.csv'
action_df = pd.read_csv(data_folder + action_file)
action_df['action_id'] = action_df['action_id'] - 1


expert_score_cols = ['score_acceptance', 'score_distraction', 'score_problem_solving', 'score_social_support']
expert_score_matrix = action_df[expert_score_cols].values

category_mapping = {"acceptance": 0, "distraction": 1, "problem_solving": 2, "social_support": 3}
action_df['category_id'] = action_df['category'].map(category_mapping)  
action_categories = action_df['category_id'].values

transition_probs = np.load(data_folder + 'functions\\2\\transition_probs_all_actions.npy')
completion_probs = np.load(data_folder + 'functions\\2\\completion_probs.npy')
reward_matrix = np.load(data_folder + 'functions\\2\\reward_matrix.npy')

NUM_ACTIONS = len(action_df)

In [4]:
env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 completion_probs=completion_probs,
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

In [5]:
num_evals = 30
pop_size = 2
num_steps_per_episode=28
total_steps = 50

model_name = f'vimorld_T={total_steps}_P={pop_size}_nO={NUM_OBJECTIVES}_nA={NUM_ACTIONS}_n_eval={num_evals}'
filename = model_name
model_file = os.path.join(results_folder, model_name, 'weights', filename)



In [6]:
pareto_archive = ParetoArchive()
seed = 42
random_state = np.random.default_rng(seed)

def adapt_weights_random(agents, num_iterations=2):
    for i in range(num_iterations):
        print("Weight adaptation cycle: ", i+1)
        for agent in agents:
            # add some random noise to the weights, ensure positivity and normalization
            noise = random_state.normal(1, 0.1, size=agent.weights.shape)
            new_weights = agent.weights * noise
            # normalize the weights
            new_weights /= np.sum(new_weights)
            agent.set_weights(new_weights)

            new_eval = agent.expected_return
            pareto_archive.add(agent.weights, new_eval)

agents = []
all_weights = random_weights(NUM_OBJECTIVES, pop_size, rng=random_state)
print("Initial weights for agents:")
print(all_weights)
for i in range(pop_size):
    weigths = all_weights[i]
    vi_agent = MOPolicyIteration(id=i, env=env.unwrapped, weights=weigths, gamma=0.9, max_iters_train=5)
    vi_agent.train()
    agents.append(vi_agent)
    pareto_archive.add(vi_agent.weights, vi_agent.expected_return)


adapt_weights_random(agents, num_iterations=2)


Initial weights for agents:
[[0.26880532 0.26120038 0.26663096 0.03128272 0.00966423 0.16241639]
 [0.20675209 0.45813669 0.01162745 0.15346431 0.01032855 0.15969091]]
Running policy iteration with weights: [0.26880532 0.26120038 0.26663096 0.03128272 0.00966423 0.16241639]
Policy converged after 16 iterations.
Running policy iteration with weights: [0.20675209 0.45813669 0.01162745 0.15346431 0.01032855 0.15969091]
Policy converged after 14 iterations.
Weight adaptation cycle:  1
Running policy iteration with weights: [0.26378047 0.28333997 0.2720824  0.02787603 0.00976878 0.14315235]
Policy converged after 4 iterations.
Updated weights to [0.26378047 0.28333997 0.2720824  0.02787603 0.00976878 0.14315235], re-computed policy.
Running policy iteration with weights: [0.22401703 0.45403097 0.01136697 0.14244397 0.01154502 0.15659604]
Policy converged after 4 iterations.
Updated weights to [0.22401703 0.45403097 0.01136697 0.14244397 0.01154502 0.15659604], re-computed policy.
Weight adap

In [7]:
for i in range(len(pareto_archive.individuals)):
    weights = pareto_archive.individuals[i]
    eval = pareto_archive.evaluations[i]
    print(f"Policy {i}: Weights: {weights}, Evaluation: {eval}")
    

Policy 0: Weights: [0.26880532 0.26120038 0.26663096 0.03128272 0.00966423 0.16241639], Evaluation: [7.53106173 7.49378532 9.07648908 2.59963508 6.56832159 9.84002654]
Policy 1: Weights: [0.20675209 0.45813669 0.01162745 0.15346431 0.01032855 0.15969091], Evaluation: [6.42701144 8.18316645 8.1489686  2.55919346 6.97422244 9.63680143]
Policy 2: Weights: [0.26378047 0.28333997 0.2720824  0.02787603 0.00976878 0.14315235], Evaluation: [7.52651812 7.50535819 9.079567   2.59921659 6.56283683 9.83959393]
Policy 3: Weights: [0.22401703 0.45403097 0.01136697 0.14244397 0.01154502 0.15659604], Evaluation: [7.30671564 7.69836627 9.00022254 2.6020177  6.40564078 9.81036477]
Policy 4: Weights: [0.25228105 0.27314503 0.2863375  0.02887174 0.01016387 0.1492008 ], Evaluation: [7.5226188  7.50800448 9.08063623 2.60019542 6.56822664 9.84081322]
Policy 5: Weights: [0.26260785 0.42054788 0.01041256 0.12633692 0.01183325 0.16826155], Evaluation: [7.37556205 7.66215015 8.99315395 2.600799   6.42982515 9.81

### Run simulations

In [9]:
def simulate(env, num_users, policy=None, verbose=False, T=28):
    data_list = []
    for user in range(num_users):
        t = 0
        obs, _ = env.reset()  # later use initial state distribution
        done = False
        while not done and t < T:
            state_idx = utils.state_to_idx(obs, NUM_VALS, MAX_COUNT)
            action = policy[state_idx] if policy is not None else env.action_space.sample()
            obs_next, rewards, terminated, truncated, info = env.step(action)
            if verbose:
                print(f"Action: {action}, Rewards: {rewards}, Info: {info}")
            user_row = {
                'user': user,
                't': t,
                'action': action,
                'state': obs,
                'next_state': obs_next,
                'counts': obs[3:],
                'rewards': rewards
            }
            data_list.append(user_row)

            done = terminated or truncated
            obs = obs_next
            t += 1
    simulation_results = pd.DataFrame(data_list)
    return simulation_results

In [10]:
NUM_USERS = 50
NUM_TIMESTEPS = 28

objectives = ["Time Required", "Fun", "Perceived Usefulness", "Expert Usefulness", "Diversity", "Completion Rate"]

In [12]:
policies = []
for weights in pareto_archive.individuals[:5]:
    vi_agent = MOPolicyIteration(id=0, env=env.unwrapped, weights=weights, gamma=0.9)
    vi_agent.train()
    policies.append(vi_agent.policy_table)
print("Number of policies:", len(policies))

Running policy iteration with weights: [0.26880532 0.26120038 0.26663096 0.03128272 0.00966423 0.16241639]
Policy converged after 8 iterations.
Running policy iteration with weights: [0.20675209 0.45813669 0.01162745 0.15346431 0.01032855 0.15969091]
Policy converged after 8 iterations.
Running policy iteration with weights: [0.26378047 0.28333997 0.2720824  0.02787603 0.00976878 0.14315235]
Policy converged after 9 iterations.
Running policy iteration with weights: [0.22401703 0.45403097 0.01136697 0.14244397 0.01154502 0.15659604]
Policy converged after 8 iterations.
Running policy iteration with weights: [0.25228105 0.27314503 0.2863375  0.02887174 0.01016387 0.1492008 ]


KeyboardInterrupt: 

In [ ]:
save_path = results_folder + model_name + '/simulations/'
save_path = os.path.join(results_folder, model_name, 'simulations')

all_sim_results =[]
if os.path.exists(save_path):
    for i in range(len(policies)):
        file_path = os.path.join(save_path, f'simulation_{i}.pkl')
        df = pd.read_pickle(file_path)
        all_sim_results.append(df)
else:
    os.makedirs(save_path)
    for i, policy in enumerate(policies):
        print(f"Simulating Policy {i}:")
        simulation_results = simulate(env, NUM_USERS, policy=policy)
        all_sim_results.append(simulation_results)
        save_file = f'simulation_{i}.pkl'
        save_to = os.path.join(save_path, save_file)
        simulation_results.to_pickle(save_to)

In [ ]:
random_sim_file = f'random/simulation_random_n={NUM_USERS}.pkl'
random_sim_path = os.path.join(results_folder, random_sim_file)

if os.path.exists(random_sim_path):
    simulation_random = pd.read_pickle(random_sim_path)
else:
    simulation_random = simulate(env, NUM_USERS)
    simulation_random.to_pickle(random_sim_path)

### Analyze and visualize simulations

In [ ]:
all_policies = all_sim_results + [simulation_random]
policy_names = [f"Policy {i}" for i in range(len(all_sim_results))] + ["Random Policy"]

for j in range(NUM_OBJECTIVES):
    print(f"\n--- {objectives[j]} ---")
    for idx, sim_res in enumerate(all_policies):
        raw_rewards = np.stack(sim_res['rewards'])
        rewards = raw_rewards.reshape(NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)
        
        user_means = np.mean(rewards[:, :, j], axis=1) 
        grand_mean = np.mean(user_means)
        n = len(user_means)
        sem = stats.sem(user_means)
        ci = stats.t.interval(0.95, df=n-1, loc=grand_mean, scale=sem)
        
        print(f"{policy_names[idx]:<15} | Mean: {grand_mean:.4f} | 95% CI: ({ci[0]:.4f}, {ci[1]:.4f})")


--- Time Required ---
Policy 0        | Mean: 0.7520 | 95% CI: (0.7404, 0.7636)
Policy 1        | Mean: 0.6808 | 95% CI: (0.6625, 0.6991)
Policy 2        | Mean: 0.8163 | 95% CI: (0.8023, 0.8302)
Policy 3        | Mean: 0.7363 | 95% CI: (0.7242, 0.7484)
Policy 4        | Mean: 0.6963 | 95% CI: (0.6835, 0.7092)
Random Policy   | Mean: 0.0393 | 95% CI: (0.0312, 0.0474)

--- Fun ---
Policy 0        | Mean: 0.7944 | 95% CI: (0.7810, 0.8077)
Policy 1        | Mean: 0.8076 | 95% CI: (0.7962, 0.8191)
Policy 2        | Mean: 0.5943 | 95% CI: (0.5690, 0.6197)
Policy 3        | Mean: 0.7691 | 95% CI: (0.7540, 0.7842)
Policy 4        | Mean: 0.8195 | 95% CI: (0.8064, 0.8327)
Random Policy   | Mean: 0.0415 | 95% CI: (0.0334, 0.0497)

--- Perceived Usefulness ---
Policy 0        | Mean: 0.9620 | 95% CI: (0.9539, 0.9701)
Policy 1        | Mean: 0.9268 | 95% CI: (0.9139, 0.9398)
Policy 2        | Mean: 0.9505 | 95% CI: (0.9388, 0.9622)
Policy 3        | Mean: 0.9511 | 95% CI: (0.9420, 0.9603)
Policy

In [ ]:
def plot_objective(objective_idx, is_cumulative):
    plt.figure(figsize=(10,6)) 
    obj_name = objectives[objective_idx]
    for idx, sim_res in enumerate(all_policies):
        name = policy_names[idx]
        raw_rewards = np.stack(sim_res['rewards'])
        rewards = raw_rewards.reshape(NUM_USERS, NUM_TIMESTEPS, NUM_OBJECTIVES)
        data = rewards[:, :, objective_idx]
        if is_cumulative:
            data = np.cumsum(data, axis=1)
        mean_rewards = np.mean(data, axis=0)
        style = '--' if "Random" in name else '-'
        plt.plot(mean_rewards, label=name, linestyle=style)

    plt.title(f"Average {obj_name} Over Time {'(Cumulative)' if is_cumulative else ''}")
    plt.xlabel("Time Step")
    plt.ylabel("Mean Reward")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>

In [ ]:
def value_iteration_factored(env, weights, gamma=0.9, theta=1e-6, max_iterations=1000):
        n_u = env.nS_user
        n_c = env.nS_count
        V = np.zeros((n_u, n_c))  
        
        # Pre-calculate scalarized rewards R[u, c, a]
        R = (env.R * weights).sum(axis=2).reshape(n_u, n_c, env.nA)
        
        # Pre-extract completion probabilities p_c[u, c, a]
        # (Assuming the last objective is completion probability)
        P_comp = env.P_comp.reshape(n_u, n_c, env.nA)

        Q = np.zeros((env.nA, n_u, n_c))
        
        for _ in range(max_iterations):
            V_old = V.copy()
            
            for a in range(env.nA):
                # User state transition 
                # P_user_a is (n_u, n_u)
                P_user_a = env.P_user[:, a, :] 
                
                # Count state transition 
                # Probability of staying vs probability of incrementing
                p_c = P_comp[:, :, a] # Shape (n_u, n_c)

                k = env.action_categories[a]  # Category of the current action
                next_indices_k = env.next_indices[k]  # Shape (n_c,)
                
                # We calculate the expected future value for all user and count states at once
                # Values for staying in same count state
                V_next_stay = P_user_a @ V_old 
                # Values for incrementing count state
                V_next_increment = V_next_stay[:, next_indices_k]
                
                # Combine based on completion probability
                Q[a] = R[:, :, a] + gamma * ((1 - p_c) * V_next_stay + p_c * V_next_increment)

            V = np.max(Q, axis=0)
            if np.max(np.abs(V - V_old)) < theta:
                break
                
        return V, np.argmax(Q, axis=0).flatten()

In [ ]:
# use single-objective RL with equal weights as baseline to compare
weights = np.array([0, 0, 0, 1.0, 0, 0])


env = mo_gym.make('user_env', num_actions=NUM_ACTIONS, 
                 num_objectives = NUM_OBJECTIVES, 
                 expert_score_matrix=expert_score_matrix, 
                 transition_probs=transition_probs, 
                 completion_probs=completion_probs,
                 reward_matrix=reward_matrix, 
                 action_categories=action_categories,
                 MAX_COUNT=MAX_COUNT)

V, policy_vi = value_iteration_factored(env.unwrapped, weights=weights)

In [ ]:
def simulate(env, num_users, policy=None, T=28):
    data_list = []
    for user in range(num_users):
        t = 0
        obs, _ = env.reset()  # later use initial state distribution
        done = False
        while not done and t < T:
            state_idx = utils.state_to_idx(tuple(obs), max_count=MAX_COUNT)
            action = policy[state_idx] if policy is not None else env.action_space.sample()
            obs_next, rewards, terminated, truncated, info = env.step(action)
            user_row = {
                'user': user,
                't': t,
                'action': action,
                'state': obs,
                'next_state': obs_next,
                'counts': obs[3:],
                'rewards': rewards
            }
            data_list.append(user_row)

            done = terminated or truncated
            obs = obs_next
            t += 1
    simulation_results = pd.DataFrame(data_list)
    return simulation_results

In [ ]:
simulation_vi = simulate(env, 50, policy_vi)


In [ ]:
all_policies = all_policies + [simulation_vi]
policy_names = policy_names + ["Policy VI"]
interact(
    plot_objective, 
    objective_idx=widgets.Dropdown(
        options=[(name, i) for i, name in enumerate(objectives)],
        value=0,
        description='Objective:',
    ),
    is_cumulative=widgets.Checkbox(
        value=True,
        description='Cumulative Sum',
    )
)

interactive(children=(Dropdown(description='Objective:', options=(('Time Required', 0), ('Fun', 1), ('Perceive…

<function __main__.plot_objective(objective_idx, is_cumulative)>